#Initialization

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Load & Rename Silver Data

In [0]:
df = spark.sql("SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM pcat.silver.orders;")

df.show(5)

#Create Gold Table & Merge Into Gold

In [0]:
if not (spark.catalog.tableExists("pcat.gold.sb_fact_orders")):
    print("creating New Fact Table")
    df.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable("pcat.gold.sb_fact_orders")
else:
    gold_delta = DeltaTable.forName(spark, "pcat.gold.sb_fact_orders")
    gold_delta.alias("source").merge(df.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Identify Affected Months

In [0]:
df_child = spark.sql("SELECT date, product_code, customer_code, sold_quantity FROM pcat.gold.sb_fact_orders")
df_child.show(10)

#Calculate Monthly Aggregates

In [0]:
df_monthly = (
    df_child
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(10, truncate=False)

#Merge Monthly Aggregates into Parent Gold

In [0]:
gold_parent_delta = DeltaTable.forName(spark, "pcat.gold.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()